In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

RAW = Path("../data/raw")

train        = pd.read_csv(RAW / "train.csv", parse_dates=["date"])
test         = pd.read_csv(RAW / "test.csv", parse_dates=["date"])
stores       = pd.read_csv(RAW / "stores.csv")
oil          = pd.read_csv(RAW / "oil.csv", parse_dates=["date"])
holidays     = pd.read_csv(RAW / "holidays_events.csv", parse_dates=["date"])
transactions = pd.read_csv(RAW / "transactions.csv", parse_dates=["date"])

tables = {"train": train, "test": test, "stores": stores,
          "oil": oil, "holidays": holidays, "transactions": transactions}

for name, df in tables.items():
    print(f"{name:13s} shape={df.shape}")

train         shape=(3000888, 6)
test          shape=(28512, 5)
stores        shape=(54, 5)
oil           shape=(1218, 2)
holidays      shape=(350, 6)
transactions  shape=(83488, 3)


In [2]:
for name, df in tables.items():
    print(f"\n{'='*50}\n{name.upper()}  {df.shape}\n{'='*50}")
    print(df.dtypes.to_string())
    print("\nMissing values:")
    print(df.isna().sum().to_string())
    print("\nHead:")
    display(df.head(3))


TRAIN  (3000888, 6)
id                      int64
date           datetime64[us]
store_nbr               int64
family                    str
sales                 float64
onpromotion             int64

Missing values:
id             0
date           0
store_nbr      0
family         0
sales          0
onpromotion    0

Head:


,id,date,store_nbr,family,sales,onpromotion
0,0,2013-01-01,1,AUTOMOTIVE,0.0,0
1,1,2013-01-01,1,BABY CARE,0.0,0
2,2,2013-01-01,1,BEAUTY,0.0,0



TEST  (28512, 5)
id                      int64
date           datetime64[us]
store_nbr               int64
family                    str
onpromotion             int64

Missing values:
id             0
date           0
store_nbr      0
family         0
onpromotion    0

Head:


,id,date,store_nbr,family,onpromotion
0,3000888,2017-08-16,1,AUTOMOTIVE,0
1,3000889,2017-08-16,1,BABY CARE,0
2,3000890,2017-08-16,1,BEAUTY,2



STORES  (54, 5)
store_nbr    int64
city           str
state          str
type           str
cluster      int64

Missing values:
store_nbr    0
city         0
state        0
type         0
cluster      0

Head:


,store_nbr,city,state,type,cluster
0,1,Quito,Pichincha,D,13
1,2,Quito,Pichincha,D,13
2,3,Quito,Pichincha,D,8


... (2 outputs left out)


HOLIDAYS  (350, 6)
date           datetime64[us]
type                      str
locale                    str
locale_name               str
description               str
transferred              bool

Missing values:
date           0
type           0
locale         0
locale_name    0
description    0
transferred    0

Head:


,date,type,locale,locale_name,description,transferred
0,2012-03-02,Holiday,Local,Manta,Fundacion de Manta,False
1,2012-04-01,Holiday,Regional,Cotopaxi,Provincializacion de Cotopaxi,False
2,2012-04-12,Holiday,Local,Cuenca,Fundacion de Cuenca,False



TRANSACTIONS  (83488, 3)
date            datetime64[us]
store_nbr                int64
transactions             int64

Missing values:
date            0
store_nbr       0
transactions    0

Head:


,date,store_nbr,transactions
0,2013-01-01,25,770
1,2013-01-02,1,2111
2,2013-01-02,2,2358


In [3]:
# Date ranges
for name in ["train", "test", "oil", "holidays", "transactions"]:
    d = tables[name]["date"]
    print(f"{name:13s} {d.min().date()} -> {d.max().date()}  unique days: {d.nunique()}")

# Train mein kitne stores/families
print("\nstores:", train["store_nbr"].nunique(), "| families:", train["family"].nunique())

# Duplicates check
print("train duplicate (date, store, family):", train.duplicated(["date", "store_nbr", "family"]).sum())
print("transactions duplicate (date, store):", transactions.duplicated(["date", "store_nbr"]).sum())

# Train mein kitne din hain aur kitne hone chahiye the (gaps pata chalenge)
full_days = pd.date_range(train["date"].min(), train["date"].max())
print("\ntrain days present:", train["date"].nunique(), "| calendar days:", len(full_days))
print("missing days:", sorted(set(full_days) - set(train["date"].unique())))

train         2013-01-01 -> 2017-08-15  unique days: 1684
test          2017-08-16 -> 2017-08-31  unique days: 16
oil           2013-01-01 -> 2017-08-31  unique days: 1218
holidays      2012-03-02 -> 2017-12-26  unique days: 312
transactions  2013-01-01 -> 2017-08-15  unique days: 1682

stores: 54 | families: 33
train duplicate (date, store, family): 0
transactions duplicate (date, store): 0

train days present: 1684 | calendar days: 1688
missing days: [Timestamp('2013-12-25 00:00:00'), Timestamp('2014-12-25 00:00:00'), Timestamp('2015-12-25 00:00:00'), Timestamp('2016-12-25 00:00:00')]


In [4]:
# 1. Oil missing
print("oil NaN:", oil["dcoilwtico"].isna().sum(), "of", len(oil))

# 2. Holidays breakdown
print("\nholiday types:\n", holidays["type"].value_counts().to_string())
print("\nlocale:\n", holidays["locale"].value_counts().to_string())
print("\ntransferred:\n", holidays["transferred"].value_counts().to_string())
print("dates with >1 event:", (holidays["date"].value_counts() > 1).sum())

# 3. Transactions gaps: kaun se store-days missing hain
full = (train[["date", "store_nbr"]].drop_duplicates()
        .merge(transactions, on=["date", "store_nbr"], how="left"))
print("\nstore-days without transactions:", full["transactions"].isna().sum())

# in missing store-days par sales kya thi?
daily_sales = train.groupby(["date", "store_nbr"])["sales"].sum().reset_index()
chk = full.merge(daily_sales, on=["date", "store_nbr"])
miss = chk[chk["transactions"].isna()]
print("missing-tx days with total sales == 0:", (miss["sales"] == 0).sum(), "of", len(miss))

oil NaN: 43 of 1218

holiday types:
 type
Holiday       221
Event          56
Additional     51
Transfer       12
Bridge          5
Work Day        5

locale:
 locale
National    174
Local       152
Regional     24

transferred:
 transferred
False    338
True      12
dates with >1 event: 31

store-days without transactions: 7448
missing-tx days with total sales == 0: 7330 of 7448


In [5]:
daily_sales = train.groupby(["date", "store_nbr"])["sales"].sum().reset_index()
full = (daily_sales.merge(transactions, on=["date", "store_nbr"], how="left"))

odd = full[full["transactions"].isna() & (full["sales"] > 0)]
print(len(odd))
print(odd.groupby("store_nbr").size().sort_values(ascending=False).head(10).to_string())
print(odd["date"].dt.to_period("M").value_counts().sort_index().to_string())

118
store_nbr
7     4
10    4
8     3
4     3
3     3
14    3
15    3
17    3
6     3
9     3
date
2013-06      4
2014-01      1
2014-03      1
2016-01    110
2016-09      2
Freq: M


In [6]:
print(holidays[holidays["type"].isin(["Transfer", "Bridge", "Work Day"])]
      .sort_values("date")[["date","type","locale","locale_name","description","transferred"]]
      .to_string())

print(holidays[holidays["type"] == "Event"]["description"].value_counts().head(15).to_string())

          date      type    locale locale_name                             description  transferred
20  2012-10-12  Transfer  National     Ecuador     Traslado Independencia de Guayaquil        False
35  2012-12-24    Bridge  National     Ecuador                          Puente Navidad        False
39  2012-12-31    Bridge  National     Ecuador               Puente Primer dia del ano        False
42  2013-01-05  Work Day  National     Ecuador                 Recupero puente Navidad        False
43  2013-01-12  Work Day  National     Ecuador      Recupero puente primer dia del ano        False
73  2013-10-11  Transfer  National     Ecuador     Traslado Independencia de Guayaquil        False
136 2014-10-10  Transfer  National     Ecuador     Traslado Independencia de Guayaquil        False
149 2014-12-20  Work Day  National     Ecuador                 Recupero Puente Navidad        False
156 2014-12-26    Bridge  National     Ecuador                          Puente Navidad        False


In [7]:
jan = odd[odd["date"].dt.to_period("M") == "2016-01"]
print(jan.groupby("date").size().to_string())

date
2016-01-01     1
2016-01-02    17
2016-01-03    53
2016-01-04    39


In [8]:
cal = pd.date_range("2013-01-01", "2017-08-31", freq="D")
oil_clean = (oil.set_index("date")
                .reindex(cal)
                .rename_axis("date"))
oil_clean["dcoilwtico"] = oil_clean["dcoilwtico"].ffill().bfill()  # bfill sirf pehle din ke liye kaam karega
oil_clean = oil_clean.reset_index()

print(oil_clean.shape)                      # (1704, 2) aana chahiye
print("NaN left:", oil_clean["dcoilwtico"].isna().sum())
print(oil_clean.head(3))

(1704, 2)
NaN left: 0
        date  dcoilwtico
0 2013-01-01       93.14
1 2013-01-02       93.14
2 2013-01-03       92.97


In [9]:
daily = (train.groupby(["date", "store_nbr"], as_index=False)["sales"].sum()
              .rename(columns={"sales": "store_sales"}))
tx = daily.merge(transactions, on=["date", "store_nbr"], how="left")
tx = tx.sort_values(["store_nbr", "date"]).reset_index(drop=True)

closed = tx["transactions"].isna() & (tx["store_sales"] == 0)
gap    = tx["transactions"].isna() & (tx["store_sales"] > 0)
tx["tx_missing"] = gap.astype("int8")          # flag sirf asli gaps par

# pichle 7 rows ka median (NaN skip hote hain), current row shamil nahi
med7 = (tx.groupby("store_nbr")["transactions"]
          .transform(lambda s: s.shift(1).rolling(7, min_periods=1).median()))

tx.loc[gap, "transactions"] = med7[gap]
tx.loc[closed, "transactions"] = 0

# agar kisi store ke bilkul shuru mein gap ho toh store ka overall median
store_med = tx.groupby("store_nbr")["transactions"].transform("median")
tx["transactions"] = tx["transactions"].fillna(store_med)

print("tx_missing flagged:", tx["tx_missing"].sum())   # 118 aana chahiye
print("NaN left:", tx["transactions"].isna().sum())
print(tx[(tx["store_nbr"] == 2) & tx["date"].between("2016-01-01", "2016-01-04")])

tx_missing flagged: 118
NaN left: 0
           date  store_nbr  store_sales  transactions  tx_missing
2776 2016-01-01          2        0.000           0.0           0
2777 2016-01-02          2    16625.706           6.0           0
2778 2016-01-03          2    23041.149        2107.0           1
2779 2016-01-04          2    18887.388        2158.0           1


In [10]:
live = tx[tx["store_sales"] > 0].copy()
live["spt"] = live["store_sales"] / live["transactions"].replace(0, np.nan)   # sales per transaction
live["store_med_tx"] = live.groupby("store_nbr")["transactions"].transform("median")
live["tx_ratio"] = live["transactions"] / live["store_med_tx"]

sus = live[live["tx_ratio"] < 0.1]     # store ke median ka 10% se bhi kam
print("suspicious rows:", len(sus))
print(sus["date"].dt.to_period("M").value_counts().sort_index().to_string())
print(sus.sort_values("tx_ratio").head(10)[["date","store_nbr","store_sales","transactions","tx_ratio"]])

suspicious rows: 6
date
2016-01    2
2016-04    2
2016-09    1
2017-04    1
Freq: M
            date  store_nbr   store_sales  transactions  tx_ratio
2777  2016-01-02          2  16625.706000           6.0  0.003176
72070 2016-09-07         43     26.000000           5.0  0.003882
1095  2016-01-04          1  14619.339020          10.0  0.005726
88767 2016-04-17         53    257.190000          33.0  0.034591
88768 2016-04-18         53    279.728000          54.0  0.056604
75645 2017-04-03         45   2741.897999         292.0  0.083049


In [11]:
live["store_med_sales"] = live.groupby("store_nbr")["store_sales"].transform("median")
live["sales_ratio"] = live["store_sales"] / live["store_med_sales"]

glitch_idx = live[(live["tx_ratio"] < 0.1) & (live["sales_ratio"] > 0.5)].index
print("glitch rows:", len(glitch_idx))
print(live.loc[glitch_idx, ["date","store_nbr","store_sales","transactions","sales_ratio"]])

# glitch values ko NaN karo, phir wahi 7-day median se impute
tx.loc[glitch_idx, "transactions"] = np.nan
med7 = (tx.groupby("store_nbr")["transactions"]
          .transform(lambda s: s.shift(1).rolling(7, min_periods=1).median()))
tx.loc[glitch_idx, "transactions"] = med7[glitch_idx]
tx.loc[glitch_idx, "tx_missing"] = 1

print("tx_missing total:", tx["tx_missing"].sum())   # 118 + glitch rows
print("NaN left:", tx["transactions"].isna().sum())

glitch rows: 2
           date  store_nbr  store_sales  transactions  sales_ratio
1095 2016-01-04          1  14619.33902          10.0     1.713548
2777 2016-01-02          2  16625.70600           6.0     1.321222
tx_missing total: 120
NaN left: 0


In [12]:
print(stores.head(3).to_string())
print("\nholiday locale_name (Local):", sorted(holidays.query("locale=='Local'")["locale_name"].unique()))
print("holiday locale_name (Regional):", sorted(holidays.query("locale=='Regional'")["locale_name"].unique()))
print("\nstore cities:", sorted(stores["city"].unique()))
print("store states:", sorted(stores["state"].unique()))

   store_nbr   city      state type  cluster
0          1  Quito  Pichincha    D       13
1          2  Quito  Pichincha    D       13
2          3  Quito  Pichincha    D        8

holiday locale_name (Local): ['Ambato', 'Cayambe', 'Cuenca', 'El Carmen', 'Esmeraldas', 'Guaranda', 'Guayaquil', 'Ibarra', 'Latacunga', 'Libertad', 'Loja', 'Machala', 'Manta', 'Puyo', 'Quevedo', 'Quito', 'Riobamba', 'Salinas', 'Santo Domingo']
holiday locale_name (Regional): ['Cotopaxi', 'Imbabura', 'Santa Elena', 'Santo Domingo de los Tsachilas']

store cities: ['Ambato', 'Babahoyo', 'Cayambe', 'Cuenca', 'Daule', 'El Carmen', 'Esmeraldas', 'Guaranda', 'Guayaquil', 'Ibarra', 'Latacunga', 'Libertad', 'Loja', 'Machala', 'Manta', 'Playas', 'Puyo', 'Quevedo', 'Quito', 'Riobamba', 'Salinas', 'Santo Domingo']
store states: ['Azuay', 'Bolivar', 'Chimborazo', 'Cotopaxi', 'El Oro', 'Esmeraldas', 'Guayas', 'Imbabura', 'Loja', 'Los Rios', 'Manabi', 'Pastaza', 'Pichincha', 'Santa Elena', 'Santo Domingo de los Tsachilas'

In [13]:
# 1. Store x date grid (train + test period)
cal = pd.date_range("2013-01-01", "2017-08-31", freq="D")
grid = (pd.MultiIndex.from_product([cal, stores["store_nbr"]], names=["date", "store_nbr"])
          .to_frame(index=False)
          .merge(stores[["store_nbr", "city", "state"]], on="store_nbr", how="left"))

# 2. Sirf asli chhutti wale rows (Work Day, Event, transferred=True bahar)
HOLIDAY_TYPES = ["Holiday", "Transfer", "Additional", "Bridge"]
hol = holidays[(holidays["transferred"] == False) & holidays["type"].isin(HOLIDAY_TYPES)]

nat = hol.loc[hol["locale"] == "National", ["date"]].drop_duplicates().assign(is_holiday_national=1)
reg = (hol.loc[hol["locale"] == "Regional", ["date", "locale_name"]]
          .drop_duplicates().rename(columns={"locale_name": "state"}).assign(is_holiday_regional=1))
loc = (hol.loc[hol["locale"] == "Local", ["date", "locale_name"]]
          .drop_duplicates().rename(columns={"locale_name": "city"}).assign(is_holiday_local=1))

# 3. Events (chhutti nahi, demand shock)
ev = (holidays.loc[holidays["type"] == "Event", ["date"]]
              .drop_duplicates().assign(is_event=1))

# 4. Merge flags, NaN ko 0 karo
flags = (grid.merge(nat, on="date", how="left")
             .merge(reg, on=["date", "state"], how="left")
             .merge(loc, on=["date", "city"], how="left")
             .merge(ev,  on="date", how="left"))

flag_cols = ["is_holiday_national", "is_holiday_regional", "is_holiday_local", "is_event"]
flags[flag_cols] = flags[flag_cols].fillna(0).astype("int8")
flags["is_holiday_any"] = flags[["is_holiday_national", "is_holiday_regional", "is_holiday_local"]].max(axis=1)

# 5. Sanity checks
print("grid shape:", flags.shape)                  # (92016, 9) aana chahiye = 1704 x 54
print(flags[flag_cols + ["is_holiday_any"]].sum().to_string())
print("\nEvent locales:", holidays.loc[holidays["type"] == "Event", "locale"].unique())

grid shape: (92016, 9)
is_holiday_national    4428
is_holiday_regional      31
is_holiday_local        362
is_event               2970
is_holiday_any         4816

Event locales: <ArrowStringArray>
['National']
Length: 1, dtype: str


In [14]:
n0 = len(train)  # 3,000,888

df = train.copy()

# 1. Store metadata (type -> store_type, taake baad mein holiday 'type' se confuse na ho)
stores_m = stores.rename(columns={"type": "store_type", "cluster": "store_cluster"})
df = df.merge(stores_m, on="store_nbr", how="left")
assert len(df) == n0

# 2. Oil
df = df.merge(oil_clean, on="date", how="left")
assert len(df) == n0

# 3. Transactions (cleaned) + flag
df = df.merge(tx[["date", "store_nbr", "transactions", "tx_missing"]],
              on=["date", "store_nbr"], how="left")
assert len(df) == n0

# 4. Holiday / event flags (city/state ab df mein hain, isliye flags se sirf flag columns lo)
df = df.merge(flags[["date", "store_nbr"] + flag_cols + ["is_holiday_any"]],
              on=["date", "store_nbr"], how="left")
assert len(df) == n0

# 5. Sort (lag/rolling ke liye zaroori)
df = df.sort_values(["store_nbr", "family", "date"]).reset_index(drop=True)

# 6. Checks
print("shape:", df.shape)
print("duplicates:", df.duplicated(["date", "store_nbr", "family"]).sum())
print("\nNaN counts:")
print(df.isna().sum()[lambda s: s > 0].to_string() or "none")
print("\ndtypes:\n", df.dtypes.to_string())
print("\nmemory MB:", round(df.memory_usage(deep=True).sum() / 1e6))

shape: (3000888, 18)
duplicates: 0

NaN counts:
Series([], )

dtypes:
 id                              int64
date                   datetime64[us]
store_nbr                       int64
family                            str
sales                         float64
onpromotion                     int64
city                              str
state                             str
store_type                        str
store_cluster                   int64
dcoilwtico                    float64
transactions                  float64
tx_missing                       int8
is_holiday_national              int8
is_holiday_regional              int8
is_holiday_local                 int8
is_event                         int8
is_holiday_any                   int8

memory MB: 390


In [15]:
OUT = Path("../data/processed")
OUT.mkdir(parents=True, exist_ok=True)

clean = df.copy()

# text columns -> category
for c in ["family", "city", "state", "store_type"]:
    clean[c] = clean[c].astype("category")

# numeric downcast
clean["id"]            = clean["id"].astype("int32")
clean["store_nbr"]     = clean["store_nbr"].astype("int8")
clean["onpromotion"]   = clean["onpromotion"].astype("int16")
clean["store_cluster"] = clean["store_cluster"].astype("int8")
clean["dcoilwtico"]    = clean["dcoilwtico"].astype("float32")
clean["transactions"]  = clean["transactions"].astype("float32")
# sales ko float64 hi rehne do (precision, aur target hai)

print("memory MB:", round(clean.memory_usage(deep=True).sum() / 1e6))

clean.to_parquet(OUT / "cleaned_merged.parquet", index=False, compression="zstd")

# reload check
chk = pd.read_parquet(OUT / "cleaned_merged.parquet")
print("reloaded shape:", chk.shape)
print("sales sum match:", np.isclose(chk["sales"].sum(), df["sales"].sum()))
print("file size MB:", round((OUT / "cleaned_merged.parquet").stat().st_size / 1e6))

memory MB: 126
reloaded shape: (3000888, 18)
sales sum match: True
file size MB: 20


Data Cleaning Summary

1. Merged the tables.
Combined train with stores, oil, transactions, and holiday flags into one table (3,000,888 rows, no duplicates, no NaNs).
Why: the model needs all context (store city, oil price, holiday, etc.) in a single row next to the sales value.

2. Oil prices: reindexed to a full daily calendar and forward-filled.
The raw file had only working days (1,218 rows, 43 NaN). We created all 1,704 days and filled each gap with the most recent known price.
Why last day’s value: oil prices change slowly, so yesterday’s price is a reasonable estimate. We chose forward-fill over interpolation because interpolation uses future prices to fill a gap, which is data leakage in forecasting. In real life you would only know the past. The only exception is the very first day (2013-01-01), which has no past, so it copied the next day’s value.

3. Transactions: three different cases, three different fixes.

Closed store days (7,330): sales were 0 and transactions were missing, so we set transactions = 0. Why: the store was closed, so zero is the true value, not a missing one.
Recording gaps (118 rows) and glitches (2 rows): we filled them with the median of the previous 7 days for that store. Why median instead of last day: a single bad day can be wrong (e.g. store 2 showed 6 transactions on a day with 16,625 in sales). Copying that would spread the error. The median of 7 days ignores such outliers and gives a typical value. We also only look at past days, so no leakage.
We added a tx_missing = 1 flag on these 120 rows so the model and teammates know the value is estimated, not real.

4. Holidays: converted to flags instead of merging.
Created is_holiday_national, is_holiday_regional (matched by state), is_holiday_local (matched by city), and is_event.
Why flags: some dates have multiple holiday entries, so a direct merge would duplicate sales rows. We excluded Work Day (it is a working day, not a holiday) and transferred holidays (the celebration moved to another date). Event rows (Black Friday, earthquake) are kept separate because they affect demand but are not days off.

5. Reduced file size.
Converted text columns to category and large numeric types to smaller ones: 881 MB down to 126 MB in memory, 20 MB as parquet.
Why: faster to load and easier for teammates to use.

One-line summary: we merged 6 separate tables into one clean table, filling gaps using only past information so the model never “sees the future.”